In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import pyodbc
import openpyxl

In [5]:
data = pd.read_csv('/Users/starsrain/2025_concord/yieldCurve_augmenting/raw_data/2023_2025_data_0210.csv')
fpd = pd.read_csv('/Users/starsrain/2025_concord/yieldCurve_augmenting/raw_data/2025fpd_0210.csv')


## Paidin by Origination Date

In [ ]:
data.head()

,Application_ID,PortFolioID,LoanID,Frequency,LPCampaign,OriginatedAmount,AppYear,AppMonth,AppWeek,ApplicationDate,...,TransactionDate,PmtYear,PmtMonth,Days_Since_Orig,Weeks_Since_Orig,PaymentType,Payment_Number,PaymentStatus,weeks_between_orig_now,CustType
0,104432023,5,I1532807-0,B,RZP200BA1PEP,700.0,2023,1,1,2023-01-05 11:01:28.000,...,2023-03-08 14:33:21.680,2023.0,3.0,62.0,9.0,Installment Pmt,6,R,162.0,NEW
1,104432023,5,I1532807-0,B,RZP200BA1PEP,700.0,2023,1,1,2023-01-05 11:01:28.000,...,2023-03-09 05:36:28.680,2023.0,3.0,63.0,10.0,Installment Pmt,7,R,162.0,NEW
2,104432023,5,I1532807-0,B,RZP200BA1PEP,700.0,2023,1,1,2023-01-05 11:01:28.000,...,2023-03-20 05:04:21.950,2023.0,3.0,74.0,11.0,Installment Pmt,8,R,162.0,NEW
3,104432289,5,I1532811-0,B,RTRNGD,250.0,2023,1,1,2023-01-05 11:02:39.000,...,2023-01-13 06:26:50.247,2023.0,1.0,8.0,2.0,Z,1,D,162.0,REPEAT
4,104432930,5,I1532819-0,MB,RZP120MA1PEP,800.0,2023,1,1,2023-01-05 11:05:06.000,...,2023-01-30 16:33:03.133,2023.0,1.0,25.0,4.0,Installment Pmt,1,D,162.0,NEW


In [8]:
data['CustType'].value_counts()

CustType
NEW        749745
REPEAT     552748
RENEWAL       953
Name: count, dtype: int64

In [3]:
data['CustType'].value_counts()

CustType
NEW        205848
REPEAT     169488
RENEWAL        17
Name: count, dtype: int64

In [4]:
fpd.head()

,OrigMonth,InstallmentNumber,Frequency,FPDFA_count_all,first_install_loan_count_all,FPDFA_rate
0,2025-01,1,ME,39,90,0.433333
1,2025-01,1,B,268,526,0.509506
2,2025-01,1,W,37,77,0.480519
3,2025-01,1,MB,39,82,0.475610
4,2025-01,2,W,50,80,0.625000


### Basic setup and helper functions

In [9]:
def compute_daily_yield(data):
    """
    Compute daily cumulative yield correctly:
    1. Get unique loans and their original amounts (avoid double counting)
    2. For each day since origination, sum all payments made up to that day
    3. Calculate yield as cumulative_payments / total_originated_amount
    """
    if data.empty:
        return pd.Series(dtype=float)
    
    # Get unique loans and their amounts (avoid double counting from multiple installments)
    loan_amounts = data.groupby('LoanID')['OriginatedAmount'].first()
    total_originated = loan_amounts.sum()
    
    # Group by Days_Since_Orig and sum payments for each day
    daily_payments = data.groupby('Days_Since_Orig')['PaidOffPaymentAmount'].sum()
    
    # Create cumulative sum of payments
    cumulative_payments = daily_payments.cumsum()
    
    # Calculate yield as cumulative payments / total originated
    yield_curve = cumulative_payments / total_originated
    
    return yield_curve

def prepare_df_strict(df):
    """Prepare dataframe with strict data cleaning"""
    df = df.copy()
    df = df[df['PaymentStatus'] != 'R']
    # Convert date columns
    date_cols = ['OriginationDate', 'DueDate']
    for col in date_cols:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors='coerce')
    
    # Drop rows with missing critical fields
    critical_fields = ['LoanID', 'OriginatedAmount', 'InstallmentNumber']
    initial_count = len(df)
    df = df.dropna(subset=critical_fields)
    dropped = initial_count - len(df)
    if dropped > 0:
        print(f"[prepare] Dropped {dropped} rows with missing critical fields")

    
    # Create cohort from origination date
    df['Cohort'] = df['OriginationDate'].dt.to_period('M').astype(str)
    
    # Ensure CustType exists
    if 'CustType' not in df.columns:
        df['CustType'] = 'Unknown'
    before_dedup = len(df)
    df = df.drop_duplicates(subset=['LoanID', 'InstallmentNumber', 'DueDate', 'PaidOffPaymentAmount'])
    after_dedup = len(df)
    if before_dedup != after_dedup:
        print(f"[prepare] Removed {before_dedup - after_dedup} duplicate records")

    
    return df

# Updated plotting function with proper data cleaning and filtering
def plot_frequency_yield_curves_corrected(data, months, max_days=200, annotate_days=None):
    """
    Plot yield curves with correct filtering and calculation:
    1. Apply proper data cleaning (remove 'R' status, deduplicate, etc.)
    2. Filter DueDate <= today
    3. Group by frequency and origination month  
    4. Calculate daily yield correctly
    """
    
    if annotate_days is None:
        annotate_days = [30, 60, 90, 120, 150, 180]
    
    # Step 1: Apply proper data cleaning using existing function
    print("Step 1: Cleaning data...")
    df_cleaned = prepare_df_strict(data)
    
    # Step 2: Filter out loans with DueDate > today
    today = pd.Timestamp.now().normalize()
    print(f"Step 2: Filtering DueDate <= {today.date()}")
    
    initial_count = len(df_cleaned)
    df_filtered = df_cleaned[df_cleaned['DueDate'] <= today].copy()
    print(f"Kept {len(df_filtered):,}/{initial_count:,} records with DueDate <= today")
    
    # Step 3: Filter to requested origination months
    df_filtered = df_filtered[df_filtered['Cohort'].isin(months)]
    
    # Step 4: Remove any rows with missing critical data for yield calculation
    df_filtered = df_filtered.dropna(subset=['LoanID', 'OriginatedAmount', 'PaidOffPaymentAmount', 'Days_Since_Orig', 'Frequency'])
    
    print(f"Final dataset: {len(df_filtered):,} records, {df_filtered['LoanID'].nunique():,} unique loans")
    
    # Define frequency groups
    freq_groups = {
        'Weekly': ['W'],
        'Biweekly': ['B'],
        'Monthly_Benefits': ['MB'],
        'Monthly_Expenses': ['ME']
    }
    
    fig, axes = plt.subplots(1, len(freq_groups), figsize=(6*len(freq_groups), 6), sharey=True)
    if len(freq_groups) == 1:
        axes = [axes]
    
    # Colors for months
    palette = plt.get_cmap('tab10')
    month_colors = {m: palette(i % 10) for i, m in enumerate(months)}
    
    curves = {}
    
    print(f"\nYield Curves for months: {', '.join(months)}")
    print('='*80)
    
    for ax, (freq_name, freq_codes) in zip(axes, freq_groups.items()):
        ax.set_title(f"{freq_name} Loans", fontsize=14, fontweight='bold')
        ax.set_xlabel('Days Since Origination')
        ax.grid(True, alpha=0.3)
        
        freq_has_data = False
        
        for month in months:
            # Filter for this month and frequency
            subset = df_filtered[
                (df_filtered['Cohort'] == month) & 
                (df_filtered['Frequency'].isin(freq_codes))
            ].copy()
            
            if subset.empty:
                continue
                
            loan_count = subset['LoanID'].nunique()
            freq_has_data = True
            
            # Calculate yield curve
            curve = compute_daily_yield(subset)
            curves.setdefault(month, {})[freq_name] = curve.copy()
            
            # Plot curve
            ax.plot(curve.index, curve.values, 
                   label=f"{month} (n={loan_count:,})", 
                   color=month_colors[month], linewidth=2)
            for d in annotate_days:
                if d in curve.index:
                    ax.text(d, curve[d], f"{curve[d]:.2f}", 
                           fontsize=8, color=month_colors[month], 
                           ha='center', va='bottom')
            
            # Print summary
            summary_vals = []
            for d in annotate_days:
                if d in curve.index:
                    summary_vals.append(f"{d}d={curve[d]:.3f}")
            print(f"  {freq_name:17s} | {month} | loans={loan_count:,} | " + ', '.join(summary_vals))
        
        if not freq_has_data:
            ax.text(0.5, 0.5, 'No Data', transform=ax.transAxes, 
                   ha='center', va='center', fontsize=12, color='gray')
        
        # Format axes
        ax.axhline(1.0, color='red', linestyle='--', alpha=0.5, label='Target = 1.0')
        ax.set_xlim(0, max_days)
        ax.set_ylim(0, 1.8)
        ax.set_yticks(np.arange(0, 1.9, 0.1))
        ax.legend(fontsize=10, loc='lower right')
    
    axes[0].set_ylabel('Cumulative Yield (Paid / Originated)')
    fig.suptitle('Corrected Cumulative Yield Curves by Frequency', fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    return curves


In [10]:
def create_payin_tables_by_frequency(df, cohorts, cap=1.70):
    """
    Create payin ratio tables by InstallmentNumber for different loan frequencies
    - Apply proper data cleaning (remove 'R' status, deduplicate, etc.)
    - Filter out loans with DueDate > today
    - Return three tables for Weekly, Biweekly, and Monthly (MB+ME combined)
    - NON-CUMULATIVE: Shows payin ratio for each individual installment
    Returns: Dictionary with tables for each frequency
    """
    
    # Step 1: Apply proper data cleaning
    print("Step 1: Cleaning data...")
    df_prepared = prepare_df_strict(df)
    
    # Step 2: Filter out loans with DueDate > today
    today = pd.Timestamp.now().normalize()
    print(f"Step 2: Filtering DueDate <= {today.date()}")
    
    initial_count = len(df_prepared)
    df_filtered = df_prepared[df_prepared['DueDate'] <= today].copy()
    print(f"Kept {len(df_filtered):,}/{initial_count:,} records with DueDate <= today")
    
    # Step 3: Filter to requested cohorts
    df_filtered = df_filtered[df_filtered['Cohort'].isin(cohorts)]
    print(f"Final dataset for cohorts {cohorts}: {len(df_filtered):,} records, {df_filtered['LoanID'].nunique():,} unique loans")
    
    frequency_mappings = {
        'Weekly_fpd': ['W'],
        'Biweekly_fpd': ['B'], 
        'Monthly_fpd': ['MB', 'ME']  
    }
    
    frequency_tables = {}
    
    for freq_name, freq_types in frequency_mappings.items():
        print(f"\nProcessing {freq_name}...")
        
        # Collect data for all cohorts for this frequency
        all_cohort_data = []
        
        for cohort in cohorts:
            # Filter data for this cohort and frequency types
            cohort_data = df_filtered[df_filtered['Cohort'] == cohort].copy()
            cohort_data = cohort_data[cohort_data['Frequency'].isin(freq_types)]
            
            if cohort_data.empty:
                continue
            
            # Aggregate payments by loan and installment
            per_installment = (cohort_data.groupby(['LoanID', 'InstallmentNumber'], as_index=False)
                             .agg(PaidThisInstallment=('PaidOffPaymentAmount', 'sum'),
                                  OriginatedAmount=('OriginatedAmount', 'first')))
            
            # Roll up to installment totals and calculate ratio
            installment_curve = (
                per_installment.groupby('InstallmentNumber', as_index=False)
                .agg(PaidThisInstallment=('PaidThisInstallment', 'sum'),
                     OriginatedAmount=('OriginatedAmount', 'sum'))
            )
            installment_curve['PayinRatio'] = (
                installment_curve['PaidThisInstallment'] / installment_curve['OriginatedAmount']
            ).clip(upper=cap)
            
            # Add OrigMonth (cohort) information
            installment_curve['OrigMonth'] = cohort
            
            # Add to collection
            all_cohort_data.append(installment_curve)
        
        if all_cohort_data:
            # Combine all cohorts for this frequency
            combined_table = pd.concat(all_cohort_data, ignore_index=True)
            
            # Sort by OrigMonth and InstallmentNumber
            combined_table = combined_table.sort_values(['OrigMonth', 'InstallmentNumber']).reset_index(drop=True)
            
            # Reorder columns: InstallmentNumber, OrigMonth, PayinRatio, PaidThisInstallment, OriginatedAmount
            combined_table = combined_table[['InstallmentNumber', 'OrigMonth', 'PayinRatio', 'PaidThisInstallment', 'OriginatedAmount']]
            
            frequency_tables[freq_name] = combined_table
            
            print(f"Created table for {freq_name}: {len(combined_table)} rows")
            print(f"InstallmentNumber range: {combined_table['InstallmentNumber'].min()} to {combined_table['InstallmentNumber'].max()}")
            print(f"OrigMonth values: {sorted(combined_table['OrigMonth'].unique())}")
            print(f"Sample data:")
            print(combined_table.head())
        else:
            print(f"No data found for {freq_name}")
    
    return frequency_tables

In [23]:
# Build a streamlined payin-by-installment table
cleaned = prepare_df_strict(data)
cohorts = sorted(cleaned['Cohort'].dropna().unique())

payin_tables = create_payin_tables_by_frequency(data, cohorts, cap=1.70)

# Combine into a single table with a frequency label
payin_by_installment = (
    pd.concat(
        [table.assign(Frequency=freq) for freq, table in payin_tables.items()],
        ignore_index=True
    )
    .sort_values(['Frequency', 'OrigMonth', 'InstallmentNumber'])
    .reset_index(drop=True)
)

payin_by_installment

[prepare] Removed 13399 duplicate records
Step 1: Cleaning data...
[prepare] Removed 13399 duplicate records
Step 2: Filtering DueDate <= 2026-02-02
Kept 203,122/204,622 records with DueDate <= today
Final dataset for cohorts ['2025-01', '2025-02', '2025-03', '2025-04', '2025-05', '2025-06', '2025-07', '2025-08', '2025-09', '2025-10', '2025-11', '2025-12', '2026-01', 'NaT']: 203,122 records, 47,590 unique loans

Processing Weekly_fpd...
Created table for Weekly_fpd: 261 rows
InstallmentNumber range: 1 to 25
OrigMonth values: ['2025-01', '2025-02', '2025-03', '2025-04', '2025-05', '2025-06', '2025-07', '2025-08', '2025-09', '2025-10', '2025-11', '2025-12', '2026-01']
Sample data:
   InstallmentNumber OrigMonth  PayinRatio  PaidThisInstallment  \
0                  1   2025-01    0.322834            132927.03   
1                  2   2025-01    0.286413             98712.19   
2                  3   2025-01    0.248288             72649.18   
3                  4   2025-01    0.247877  

,InstallmentNumber,OrigMonth,PayinRatio,PaidThisInstallment,OriginatedAmount,Frequency
0,1,2025-01,0.476318,1121133.20,2353750.0,Biweekly_fpd
1,2,2025-01,0.413649,678383.69,1640000.0,Biweekly_fpd
2,3,2025-01,0.377377,488042.36,1293250.0,Biweekly_fpd
3,4,2025-01,0.361419,374285.63,1035600.0,Biweekly_fpd
4,5,2025-01,0.425229,354088.39,832700.0,Biweekly_fpd
...,...,...,...,...,...,...
644,9,2025-12,0.220000,220.00,1000.0,Weekly_fpd
645,1,2026-01,0.168966,490.00,2900.0,Weekly_fpd
646,2,2026-01,0.200000,460.00,2300.0,Weekly_fpd
647,3,2026-01,0.200000,460.00,2300.0,Weekly_fpd


In [24]:
# Save output to Excel
from pathlib import Path

output_dir = Path('/Users/starsrain/2025_concord/yieldCurve_augmenting/payin_out')
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / 'payin_by_installment_v2.xlsx'
payin_by_installment.to_excel(output_path, index=False)

output_path

PosixPath('/Users/starsrain/2025_concord/yieldCurve_augmenting/payin_out/payin_by_installment_v2.xlsx')

## Paid in by install - AppDate View

In [11]:
appDate_data = pd.read_csv('/Users/starsrain/2025_concord/yieldCurve_augmenting/raw_data/2023_2025_data_0210.csv')

In [12]:
# ApplicationDate-anchored helpers using AppMonth naming

def prepare_df_strict_application(df):
    """Prepare dataframe with strict data cleaning using ApplicationDate"""
    df = df.copy()
    df = df[df['PaymentStatus'] != 'R']

    # Convert date columns
    date_cols = ['ApplicationDate', 'DueDate']
    for col in date_cols:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors='coerce')

    # Drop rows with missing critical fields
    critical_fields = ['LoanID', 'OriginatedAmount', 'InstallmentNumber']
    initial_count = len(df)
    df = df.dropna(subset=critical_fields)
    dropped = initial_count - len(df)
    if dropped > 0:
        print(f"[prepare] Dropped {dropped} rows with missing critical fields")

    # Create application month from application date
    df['AppMonth'] = df['ApplicationDate'].dt.to_period('M').astype(str)

    # Ensure CustType exists
    if 'CustType' not in df.columns:
        df['CustType'] = 'Unknown'

    before_dedup = len(df)
    df = df.drop_duplicates(subset=['LoanID', 'InstallmentNumber', 'DueDate', 'PaidOffPaymentAmount'])
    after_dedup = len(df)
    if before_dedup != after_dedup:
        print(f"[prepare] Removed {before_dedup - after_dedup} duplicate records")

    return df


def create_payin_tables_by_frequency_application(df, app_months, cap=1.70):
    """
    Create payin ratio tables by InstallmentNumber for different loan frequencies
    anchored on ApplicationDate (same processing as OriginationDate version).
    """
    # Step 1: Apply proper data cleaning
    print("Step 1: Cleaning data...")
    df_prepared = prepare_df_strict_application(df)

    # Step 2: Filter out loans with DueDate > today
    today = pd.Timestamp.now().normalize()
    print(f"Step 2: Filtering DueDate <= {today.date()}")

    initial_count = len(df_prepared)
    df_filtered = df_prepared[df_prepared['DueDate'] <= today].copy()
    print(f"Kept {len(df_filtered):,}/{initial_count:,} records with DueDate <= today")

    # Step 3: Filter to requested application months
    df_filtered = df_filtered[df_filtered['AppMonth'].isin(app_months)]
    print(f"Final dataset for months {app_months}: {len(df_filtered):,} records, {df_filtered['LoanID'].nunique():,} unique loans")

    frequency_mappings = {
        'Weekly_fpd': ['W'],
        'Biweekly_fpd': ['B'],
        'Monthly_fpd': ['MB', 'ME']
    }

    frequency_tables = {}

    for freq_name, freq_types in frequency_mappings.items():
        print(f"\nProcessing {freq_name}...")

        freq_data = df_filtered[df_filtered['Frequency'].isin(freq_types)].copy()
        if freq_data.empty:
            print(f"No data found for {freq_name}")
            continue

        # Aggregate payments by app month, cust type, loan, and installment
        per_installment = (
            freq_data.groupby(['AppMonth', 'CustType', 'LoanID', 'InstallmentNumber'], as_index=False)
            .agg(PaidThisInstallment=('PaidOffPaymentAmount', 'sum'),
                 OriginatedAmount=('OriginatedAmount', 'first'))
        )

        # Roll up to installment totals and calculate ratio by app month + cust type
        combined_table = (
            per_installment.groupby(['AppMonth', 'CustType', 'InstallmentNumber'], as_index=False)
            .agg(PaidThisInstallment=('PaidThisInstallment', 'sum'),
                 OriginatedAmount=('OriginatedAmount', 'sum'))
        )
        combined_table['PayinRatio'] = (
            combined_table['PaidThisInstallment'] / combined_table['OriginatedAmount']
        ).clip(upper=cap)

        combined_table = combined_table.sort_values(['AppMonth', 'CustType', 'InstallmentNumber']).reset_index(drop=True)
        combined_table = combined_table[['InstallmentNumber', 'AppMonth', 'CustType', 'PayinRatio', 'PaidThisInstallment', 'OriginatedAmount']]

        frequency_tables[freq_name] = combined_table

        print(f"Created table for {freq_name}: {len(combined_table)} rows")
        print(f"InstallmentNumber range: {combined_table['InstallmentNumber'].min()} to {combined_table['InstallmentNumber'].max()}")
        print(f"AppMonth values: {sorted(combined_table['AppMonth'].unique())}")
        print(f"CustType values: {sorted(combined_table['CustType'].dropna().unique())}")
        print(f"Sample data:")
        print(combined_table.head())

    return frequency_tables

In [13]:
# ApplicationDate-anchored payin table (AppMonth)
app_cleaned = prepare_df_strict_application(appDate_data)
app_months = sorted(app_cleaned['AppMonth'].dropna().unique())

app_payin_tables = create_payin_tables_by_frequency_application(appDate_data, app_months, cap=1.70)

app_payin_by_installment = (
    pd.concat(
        [table.assign(Frequency=freq) for freq, table in app_payin_tables.items()],
        ignore_index=True
    )
    .sort_values(['Frequency', 'AppMonth', 'CustType', 'InstallmentNumber'])
    .reset_index(drop=True)
)

key_cols = ['Frequency', 'AppMonth', 'CustType', 'InstallmentNumber']
print('Rows:', len(app_payin_by_installment))
print('CustType values:', sorted(app_payin_by_installment['CustType'].dropna().unique()))
print('Duplicate key rows:', app_payin_by_installment.duplicated(key_cols).sum())

app_payin_by_installment

[prepare] Removed 86557 duplicate records
Step 1: Cleaning data...
[prepare] Removed 86557 duplicate records
Step 2: Filtering DueDate <= 2026-02-12
Kept 648,277/651,953 records with DueDate <= today
Final dataset for months ['2023-01', '2023-02', '2023-03', '2023-04', '2023-05', '2023-06', '2023-07', '2023-08', '2023-09', '2023-10', '2023-11', '2023-12', '2024-01', '2024-02', '2024-03', '2024-04', '2024-05', '2024-06', '2024-07', '2024-08', '2024-09', '2024-10', '2024-11', '2024-12']: 648,277 records, 121,039 unique loans

Processing Weekly_fpd...
Created table for Weekly_fpd: 1000 rows
InstallmentNumber range: 1 to 25
AppMonth values: ['2023-01', '2023-02', '2023-03', '2023-04', '2023-05', '2023-06', '2023-07', '2023-08', '2023-09', '2023-10', '2023-11', '2023-12', '2024-01', '2024-02', '2024-03', '2024-04', '2024-05', '2024-06', '2024-07', '2024-08', '2024-09', '2024-10', '2024-11', '2024-12']
CustType values: ['NEW', 'RENEWAL', 'REPEAT']
Sample data:
   InstallmentNumber AppMonth C

,InstallmentNumber,AppMonth,CustType,PayinRatio,PaidThisInstallment,OriginatedAmount,Frequency
0,1,2023-01,NEW,0.420988,418146.30,993250.0,Biweekly_fpd
1,2,2023-01,NEW,0.388284,318839.05,821150.0,Biweekly_fpd
2,3,2023-01,NEW,0.375364,258757.16,689350.0,Biweekly_fpd
3,4,2023-01,NEW,0.349027,197008.06,564450.0,Biweekly_fpd
4,5,2023-01,NEW,0.454387,205951.12,453250.0,Biweekly_fpd
...,...,...,...,...,...,...,...
3016,16,2024-12,REPEAT,0.095238,4385.71,46050.0,Weekly_fpd
3017,17,2024-12,REPEAT,0.073753,2806.29,38050.0,Weekly_fpd
3018,18,2024-12,REPEAT,0.066304,1863.15,28100.0,Weekly_fpd
3019,19,2024-12,REPEAT,0.053364,1174.00,22000.0,Weekly_fpd


In [14]:
# Save output to Excel
from pathlib import Path

output_dir = Path('/Users/starsrain/2025_concord/yieldCurve_augmenting/payin_out')
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / 'appDate_payin_by_installment_with_custtype_v2.xlsx'
app_payin_by_installment.to_excel(output_path, index=False)

output_path

PosixPath('/Users/starsrain/2025_concord/yieldCurve_augmenting/payin_out/appDate_payin_by_installment_with_custtype_v2.xlsx')

In [31]:
# Moved to install_notebooks/jcx_cnt_by_installment_v1.ipynb

In [32]:
# Moved to install_notebooks/jcx_cnt_by_installment_v1.ipynb

(468, 5)

In [33]:
# Moved to install_notebooks/jcx_cnt_by_installment_v1.ipynb

   Year CustType Frequency  Paidoff  CountOfApplicationID
0  2023      NEW         B        0                    73
1  2023      NEW         B        1                  4787
2  2023      NEW         B        2                  2833
3  2023      NEW         B        3                  2090
4  2023      NEW         B        4                  2035


In [ ]:
# Moved to install_notebooks/jcx_cnt_by_installment_v1.ipynb

Max InstallmentNumber: 165
[prepare] Removed 86557 duplicate records
Max InstallmentNumber by Year/CustType/Frequency (sample):
    Year CustType Frequency  MaxInstallment
0   2023      NEW         B              20
1   2023      NEW         M              20
2   2023      NEW         W              20
3   2023  RENEWAL         B              18
4   2023  RENEWAL         M              16
5   2023  RENEWAL         W              10
6   2023   REPEAT         B              20
7   2023   REPEAT         M              20
8   2023   REPEAT         W              20
9   2024      NEW         B              24
10  2024      NEW         M              19
11  2024      NEW         W              25


,Application_ID,PortFolioID,LoanID,Frequency,LPCampaign,OriginatedAmount,AppYear,AppMonth,AppWeek,ApplicationDate,...,TransactionDate,PmtYear,PmtMonth,Days_Since_Orig,Weeks_Since_Orig,PaymentType,Payment_Number,PaymentStatus,weeks_between_orig_now,CustType
0,104432023,5,I1532807-0,B,RZP200BA1PEP,700.0,2023,1,1,2023-01-05 11:01:28.000,...,2023-03-08 14:33:21.680,2023.0,3.0,62.0,9.0,Installment Pmt,6,R,162.0,NEW
1,104432023,5,I1532807-0,B,RZP200BA1PEP,700.0,2023,1,1,2023-01-05 11:01:28.000,...,2023-03-09 05:36:28.680,2023.0,3.0,63.0,10.0,Installment Pmt,7,R,162.0,NEW
2,104432023,5,I1532807-0,B,RZP200BA1PEP,700.0,2023,1,1,2023-01-05 11:01:28.000,...,2023-03-20 05:04:21.950,2023.0,3.0,74.0,11.0,Installment Pmt,8,R,162.0,NEW
3,104432289,5,I1532811-0,B,RTRNGD,250.0,2023,1,1,2023-01-05 11:02:39.000,...,2023-01-13 06:26:50.247,2023.0,1.0,8.0,2.0,Z,1,D,162.0,REPEAT
4,104432930,5,I1532819-0,MB,RZP120MA1PEP,800.0,2023,1,1,2023-01-05 11:05:06.000,...,2023-01-30 16:33:03.133,2023.0,1.0,25.0,4.0,Installment Pmt,1,D,162.0,NEW


In [37]:
# Moved to install_notebooks/jcx_cnt_by_installment_v1.ipynb

,Year,CustType,Frequency,Install_num,total_loans
0,2025,NEW,B,1.0,18161
1,2025,NEW,B,2.0,15474
2,2025,NEW,B,3.0,11858
3,2025,NEW,B,4.0,9055
4,2025,NEW,B,5.0,7177
...,...,...,...,...,...
136,2025,REPEAT,W,24.0,15
137,2025,REPEAT,W,25.0,13
138,2025,REPEAT,W,26.0,13
139,2025,REPEAT,W,27.0,13


In [22]:
# Moved to install_notebooks/jcx_cnt_by_installment_v1.ipynb

In [23]:
# Moved to install_notebooks/jcx_cnt_by_installment_v1.ipynb

[prepare] Removed 86557 duplicate records
Step 1: Cleaning data...
[prepare] Removed 86557 duplicate records
Step 2: Filtering DueDate <= 2026-02-10
Kept 648,275/651,953 records with DueDate <= today
Final dataset for months ['2023-01', '2023-02', '2023-03', '2023-04', '2023-05', '2023-06', '2023-07', '2023-08', '2023-09', '2023-10', '2023-11', '2023-12', '2024-01', '2024-02', '2024-03', '2024-04', '2024-05', '2024-06', '2024-07', '2024-08', '2024-09', '2024-10', '2024-11', '2024-12']: 648,275 records, 121,039 unique loans


,Year,CustType,Frequency,Install_num,total_loans
0,2023,NEW,B,1,20705
1,2023,NEW,B,2,16292
2,2023,NEW,B,3,13413
3,2023,NEW,B,4,11439
4,2023,NEW,B,5,9247
...,...,...,...,...,...
321,2024,REPEAT,W,20,148
322,2024,REPEAT,W,21,1
323,2024,REPEAT,W,22,1
324,2024,REPEAT,W,24,1


In [38]:
# Moved to install_notebooks/jcx_cnt_by_installment_v1.ipynb

PosixPath('/Users/starsrain/2025_concord/yieldCurve_augmenting/by_installment_out/2025_raw_loan_counts_by_installment_v1.xlsx')